# Base Analítica — Tech Challenge Fase 3

**Frente:** Vinicius Moreira
**Objetivo:** materializar a base analítica no grão de aluno para a predição de alfabetização.

## Linhagem com a Fase 2

A base sai das mesmas tabelas da Base dos Dados que alimentaram a camada Gold da Fase 2.
A equivalência foi verificada numericamente: a tabela `municipio` do BigQuery reproduz o
`fato_indicador` da Gold em todas as métricas — 23.995 linhas, 5.550 municípios, 2 anos,
taxa mínima 2,12 e máxima 100,0.

## Decisões de escopo

- **Grão:** um aluno do 2º ano do EF avaliado em 2024. A Gold da Fase 2 era agregada por
  município, o que não atende a exigência de prever o aluno.
- **Alvo:** proficiência em Língua Portuguesa >= 743 pontos, critério oficial do Indicador
  Criança Alfabetizada.
- **Defasagem:** o desempenho do município entra com dados de 2023. Usar 2024 seria vazar
  a média do próprio alvo.
- **Colunas de auditoria:** `proficiencia` e `alfabetizado_fonte` são conferidas aqui e
  descartadas antes de gravar. Não chegam ao Parquet, o que torna o vazamento mais grave
  impossível por construção.

## 1. Autenticação e configuração

In [31]:
# ============================================================
# Autenticação no Google Cloud pelo Colab
# ============================================================
from google.colab import auth

auth.authenticate_user()
print("Autenticado com sucesso!")

Autenticado com sucesso!


In [32]:
# ============================================================
# Configuração
# PROJETO_GCP: o dataset é público, mas a leitura é cobrada no projeto de quem executa
# ============================================================
PROJETO_GCP = "tech-challenge-3-grupo-61"

SEMENTE = 42
FRACAO_AMOSTRA = 0.16  # ~300 mil linhas de 1,85 milhão

# Derivadas da proficiência: geram o alvo e não podem virar feature
COLUNAS_AUDITORIA = ["proficiencia", "alfabetizado_fonte"]

# Estratos da amostra: preservam a proporção por estado, rede e alvo
COLUNAS_ESTRATO = ["sigla_uf", "rede", "alfabetizado"]

print(f"Projeto configurado: {PROJETO_GCP}")

Projeto configurado: tech-challenge-3-grupo-61


## 2. Consulta

Fonte única da verdade da base. Qualquer alteração de escopo acontece aqui.

In [33]:
CONSULTA_BASE_ANALITICA = """
-- ═══════════════════════════════════════════════════════════════
-- Base analítica no grão de aluno para predição de alfabetização
-- Fonte: Base dos Dados / BigQuery — mesma origem da Gold da Fase 2
-- Grão: um aluno do 2º ano do EF avaliado em 2024
-- Alvo: alfabetizado = proficiência em LP >= 743 pontos (escala Saeb)
-- Regra: desempenho municipal vem de 2023, para não vazar o alvo de 2024
-- ═══════════════════════════════════════════════════════════════

WITH alunos_avaliados AS (
    -- Ausente ou prova não preenchida não tem proficiência: não gera alvo válido.
    SELECT
        id_aluno,
        id_escola,
        LPAD(id_municipio, 7, '0') AS id_municipio,
        rede,
        caderno,
        peso_aluno,
        proficiencia,
        alfabetizado AS alfabetizado_fonte
    FROM `basedosdados.br_inep_avaliacao_alfabetizacao.alunos`
    WHERE ano = 2024
      AND presenca = '1'
      AND preenchimento_caderno = '1'
      AND proficiencia IS NOT NULL
),

desempenho_ano_anterior AS (
    -- Desempenho do município em 2023. Não é leakage: é anterior ao alvo de 2024.
    -- rede = '5' é a linha consolidada do município, uma por município.
    SELECT
        LPAD(id_municipio, 7, '0') AS id_municipio,
        taxa_alfabetizacao         AS taxa_municipio_2023,
        media_portugues            AS media_portugues_municipio_2023
    FROM `basedosdados.br_inep_avaliacao_alfabetizacao.municipio`
    WHERE ano = 2023
      AND rede = '5'
),

metas_municipio AS (
    -- Metas pactuadas do Compromisso Criança Alfabetizada. Existem só para a rede
    -- Municipal, mas entram como atributo de contexto do território.
    SELECT
        LPAD(id_municipio, 7, '0') AS id_municipio,
        meta_alfabetizacao_2024,
        meta_alfabetizacao_2026,
        meta_alfabetizacao_2030,
        nivel_alfabetizacao,
        percentual_participacao
    FROM `basedosdados.br_inep_avaliacao_alfabetizacao.meta_alfabetizacao_municipio`
    WHERE ano = 2024
),

territorio AS (
    -- Diretório oficial de municípios do IBGE.
    SELECT DISTINCT
        LPAD(id_municipio, 7, '0') AS id_municipio,
        nome                       AS nome_municipio,
        sigla_uf,
        nome_regiao,
        nome_mesorregiao,
        capital_uf,
        amazonia_legal
    FROM `basedosdados.br_bd_diretorios_brasil.municipio`
),

populacao_recente AS (
    -- Ano mais recente disponível até 2024, um registro por município.
    SELECT
        LPAD(id_municipio, 7, '0') AS id_municipio,
        populacao
    FROM `basedosdados.br_ibge_populacao.municipio`
    WHERE ano <= 2024
    QUALIFY ROW_NUMBER() OVER (PARTITION BY id_municipio ORDER BY ano DESC) = 1
),

pib_recente AS (
    -- Os valores adicionados setoriais só estão publicados até 2021, enquanto o PIB
    -- total vai até 2023. Fixamos tudo no ano mais recente com o conjunto completo,
    -- para que razões entre pib e va_* usem números do mesmo ano.
    SELECT
        LPAD(id_municipio, 7, '0') AS id_municipio,
        pib,
        va_agropecuaria,
        va_industria,
        va_servicos,
        va_adespss
    FROM `basedosdados.br_ibge_pib.municipio`
    WHERE ano <= 2024
      AND va_agropecuaria IS NOT NULL
    QUALIFY ROW_NUMBER() OVER (PARTITION BY id_municipio ORDER BY ano DESC) = 1
)

SELECT
    -- Identificação e chave de agrupamento para a validação cruzada
    a.id_aluno,
    a.id_escola,
    a.id_municipio,

    -- Contexto educacional do aluno
    a.rede,
    a.caderno,
    a.peso_aluno,

    -- Território
    t.nome_municipio,
    t.sigla_uf,
    t.nome_regiao,
    t.nome_mesorregiao,
    t.capital_uf,
    t.amazonia_legal,

     -- Socioeconômico (população na referência mais recente; PIB e setoriais em 2021)
    p.populacao,
    pib.pib,
    pib.va_agropecuaria,
    pib.va_industria,
    pib.va_servicos,
    pib.va_adespss,

    -- Desempenho do município no ano anterior (defasagem)
    d.taxa_municipio_2023,
    d.media_portugues_municipio_2023,

    -- Metas pactuadas
    m.meta_alfabetizacao_2024,
    m.meta_alfabetizacao_2026,
    m.meta_alfabetizacao_2030,
    m.nivel_alfabetizacao,
    m.percentual_participacao,

    -- Alvo
    IF(a.proficiencia >= 743, 1, 0) AS alfabetizado,

    -- Auditoria: conferidas e descartadas antes de gravar. Não chegam ao Parquet.
    a.proficiencia,
    a.alfabetizado_fonte

FROM alunos_avaliados       AS a
LEFT JOIN desempenho_ano_anterior AS d   ON d.id_municipio   = a.id_municipio
LEFT JOIN metas_municipio         AS m   ON m.id_municipio   = a.id_municipio
LEFT JOIN territorio              AS t   ON t.id_municipio   = a.id_municipio
LEFT JOIN populacao_recente       AS p   ON p.id_municipio   = a.id_municipio
LEFT JOIN pib_recente             AS pib ON pib.id_municipio = a.id_municipio
"""

## 3. Execução

In [34]:
# ============================================================
# Executa a consulta e traz o resultado para um DataFrame
# ============================================================
from google.cloud import bigquery

cliente = bigquery.Client(project=PROJETO_GCP)

print("Executando a consulta no BigQuery...")
df = cliente.query(CONSULTA_BASE_ANALITICA).to_dataframe()

print(f"  {len(df):,} linhas retornadas")
print(f"  {df.shape[1]} colunas")
df.head()

Executando a consulta no BigQuery...
  1,851,852 linhas retornadas
  28 colunas


,id_aluno,id_escola,id_municipio,rede,caderno,peso_aluno,nome_municipio,sigla_uf,nome_regiao,nome_mesorregiao,...,taxa_municipio_2023,media_portugues_municipio_2023,meta_alfabetizacao_2024,meta_alfabetizacao_2026,meta_alfabetizacao_2030,nivel_alfabetizacao,percentual_participacao,alfabetizado,proficiencia,alfabetizado_fonte
0,21047847,60005060,2110302,3,1,1.0,Santo Antônio dos Lopes,MA,Nordeste,Centro Maranhense,...,52.71,745.4972,57.23,65.84,80.0,4,95.04,1,789.50,1
1,22007394,60007212,2208403,3,1,1.0,Piripiri,PI,Nordeste,Norte Piauiense,...,33.16,719.9747,40.06,54.82,80.0,2,97.72,1,770.47,1
2,23018751,60008998,2306900,3,1,1.0,Jaguaribe,CE,Nordeste,Jaguaribe,...,85.97,790.4814,80.00,80.00,80.0,5,99.11,1,793.25,1
3,23082543,60008897,2311405,3,1,1.0,Quixeramobim,CE,Nordeste,Sertões Cearenses,...,98.53,821.9508,80.00,80.00,80.0,5,98.87,1,818.39,1
4,27010877,60013225,2703304,3,1,1.0,Inhapi,AL,Nordeste,Sertão Alagoano,...,61.54,747.3533,67.49,72.09,80.0,5,97.29,1,775.55,1


## 4. Validação do alvo

Confere se o corte de 743 pontos reproduz o campo oficial da fonte. A célula falha de
propósito em caso de divergência: gravar uma base com regra de negócio diferente da
oficial é pior do que não gravar nada.

In [35]:
# ============================================================
# Validação: o corte de 743 tem que reproduzir o campo oficial
# ============================================================
divergencias = (df["alfabetizado"].astype(str) != df["alfabetizado_fonte"]).sum()

if divergencias > 0:
    raise ValueError(
        f"{divergencias:,} divergências entre o corte de 743 e o campo oficial. "
        "Base não gravada."
    )

print(f"Validação OK: 0 divergências em {len(df):,} linhas")
print(f"  Taxa de alfabetizados: {df['alfabetizado'].mean() * 100:.1f}%")
print(f"  Municípios distintos:  {df['id_municipio'].nunique():,}")

Validação OK: 0 divergências em 1,851,852 linhas
  Taxa de alfabetizados: 59.8%
  Municípios distintos:  5,517


## 5. Descarte das colunas de auditoria

A partir daqui `proficiencia` e `alfabetizado_fonte` deixam de existir. É a barreira de
leakage: ninguém pode usar uma coluna que não está no arquivo.

In [36]:
# ============================================================
# Remove as colunas derivadas da proficiência
# ============================================================
df_base = df.drop(columns=COLUNAS_AUDITORIA)

print(f"Colunas removidas: {COLUNAS_AUDITORIA}")
print(f"Base final: {df_base.shape[0]:,} linhas x {df_base.shape[1]} colunas")

Colunas removidas: ['proficiencia', 'alfabetizado_fonte']
Base final: 1,851,852 linhas x 26 colunas


## 6. Amostra estratificada

Estratificar por UF, rede e alvo mantém a proporção de cada grupo. Uma amostra aleatória
simples sub-representaria estados pequenos e distorceria a análise exploratória.

In [37]:
# ============================================================
# Amostra estratificada para desenvolvimento das demais frentes
# ============================================================
df_amostra = (
    df_base
    .groupby(COLUNAS_ESTRATO, group_keys=False, observed=True)
    .sample(frac=FRACAO_AMOSTRA, random_state=SEMENTE)
    .reset_index(drop=True)
)

print(f"Amostra: {len(df_amostra):,} linhas ({len(df_amostra) / len(df_base) * 100:.1f}%)")
print(f"  Taxa de alfabetizados na amostra: {df_amostra['alfabetizado'].mean() * 100:.1f}%")
print(f"  Taxa na base completa:            {df_base['alfabetizado'].mean() * 100:.1f}%")

Amostra: 296,292 linhas (16.0%)
  Taxa de alfabetizados na amostra: 59.8%
  Taxa na base completa:            59.8%


## 7. Gravação

A base completa vai para o Drive compartilhado — é grande demais para o Git. A amostra é
pequena e vai versionada em `data/`.

In [38]:
# ============================================================
# Monta o Drive para gravar a base completa direto na pasta da equipe
# ============================================================
from google.colab import drive

drive.mount("/content/drive")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [39]:
# ============================================================
# Gravação em Parquet
# ============================================================
from pathlib import Path

PASTA_DRIVE = Path("/content/drive/MyDrive/tech-challenge-fase3/data")
PASTA_LOCAL = Path("/content/data")

PASTA_DRIVE.mkdir(parents=True, exist_ok=True)
PASTA_LOCAL.mkdir(parents=True, exist_ok=True)

# Base completa: fica no Drive, fora do Git
caminho_completo = PASTA_DRIVE / "base_analitica.parquet"
df_base.to_parquet(caminho_completo, index=False, compression="snappy")

# Amostra: vai versionada no repositório
caminho_amostra = PASTA_LOCAL / "base_analitica_amostra.parquet"
df_amostra.to_parquet(caminho_amostra, index=False, compression="snappy")

print(f"Base completa: {caminho_completo}")
print(f"  {caminho_completo.stat().st_size / 1024**2:.1f} MB")
print(f"Amostra:       {caminho_amostra}")
print(f"  {caminho_amostra.stat().st_size / 1024**2:.1f} MB")

Base completa: /content/drive/MyDrive/tech-challenge-fase3/data/base_analitica.parquet
  44.5 MB
Amostra:       /content/data/base_analitica_amostra.parquet
  9.2 MB


## 8. Contrato de colunas

Saída para o dicionário de dados. É o que as demais frentes recebem.

In [40]:
# ============================================================
# Gera o contrato de colunas: tipo, nulos e cardinalidade
# ============================================================
import pandas as pd

contrato = pd.DataFrame({
    "coluna": df_base.columns,
    "tipo": [str(t) for t in df_base.dtypes],
    "nulos": df_base.isna().sum().values,
    "pct_nulos": (df_base.isna().mean() * 100).round(2).values,
    "valores_distintos": [df_base[c].nunique() for c in df_base.columns],
})

contrato.to_csv("/content/data/contrato_colunas.csv", index=False)
contrato

,coluna,tipo,nulos,pct_nulos,valores_distintos
0,id_aluno,object,0,0.00,1851852
1,id_escola,object,0,0.00,42328
2,id_municipio,object,0,0.00,5517
3,rede,object,0,0.00,3
4,caderno,object,0,0.00,22
5,peso_aluno,float64,0,0.00,540
6,nome_municipio,object,0,0.00,5249
7,sigla_uf,object,0,0.00,26
8,nome_regiao,object,0,0.00,5
9,nome_mesorregiao,object,0,0.00,135
